In [1]:
# Cellule 2 : Script de génération de la carte démographique
import json
import os
from pathlib import Path
import openpyxl

# -------------------------------------------------------------
# 1. Chemins d'accès dans le Lakehouse Microsoft Fabric
# -------------------------------------------------------------
LAKEHOUSE_PATH = Path("/lakehouse/default/Files")

EXCEL_PATH   = LAKEHOUSE_PATH / "silver" / "insee_population" / "projections_scenario_central_effectifs.xlsx"
GEOJSON_PATH = LAKEHOUSE_PATH / "silver" / "insee_population" / "departements.geojson"

OUTPUT_DIR   = LAKEHOUSE_PATH / "gold" / "maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML  = OUTPUT_DIR / "carte_demographie_france.html"

# Dictionnaire de correction des accents corrompus dans l'Excel Insee
names_clean = {
    "Bouches-du-Rhne": "Bouches-du-Rhône", "Cte-d'Or": "Côte-d'Or",
    "Ctes-d'Armor": "Côtes-d'Armor", "Drme": "Drôme", "Finistre": "Finistère",
    "Hrault": "Hérault", "Isre": "Isère", "Nivre": "Nièvre",
    "Puy-de-Dme": "Puy-de-Dôme", "Pyrnes-Atlantiques": "Pyrénées-Atlantiques",
    "Hautes-Pyrnes": "Hautes-Pyrénées", "Pyrnes-Orientales": "Pyrénées-Orientales",
    "Rhne": "Rhône", "Haute-Sane": "Haute-Saône", "Sane-et-Loire": "Saône-et-Loire",
    "Deux-Svres": "Deux-Sèvres", "Vende": "Vendée", "Corrze": "Corrèze",
    "Ardche": "Ardèche", "Lozre": "Lozère", "Arige": "Ariège",
    "Val-d'Oise": "Val-d'Oise", "La Runion": "La Réunion",
}

print(f"Lecture des données depuis : {EXCEL_PATH}")
wb = openpyxl.load_workbook(str(EXCEL_PATH), data_only=True)
s_pop = wb["pop_departements"]
s_age = wb["effectifs_age_departements"]

# -------------------------------------------------------------
# 2. Extraction des données démographiques
# -------------------------------------------------------------
pop_data = {}

# Feuille 1 : Population totale (2026 vs 2040)
for r in range(8, s_pop.max_row + 1):
    code = str(s_pop.cell(r, 1).value or "").strip()
    if not code or code.lower() == "none":
        continue
    if len(code) == 1:
        code = "0" + code
    
    nom = names_clean.get(str(s_pop.cell(r, 2).value or "").strip(), str(s_pop.cell(r, 2).value or "").strip())
    p26 = float(s_pop.cell(r, 22).value or 0) * 1000  # col 22 = 2026
    p40 = float(s_pop.cell(r, 36).value or 0) * 1000  # col 36 = 2040
    evol_abs = p40 - p26
    evol_pct = round((evol_abs / p26) * 100, 2) if p26 > 0 else 0
    
    pop_data[code] = {
        "dep": code,
        "nom": nom,
        "pop_2026": int(p26),
        "pop_2040": int(p40),
        "evol_pop_abs": int(evol_abs),
        "evol_pop_pct": evol_pct
    }

# Feuille 2 : Effectifs 20-74 ans en 2040 (colonnes 31 à 41)
cols_20_74_40 = range(31, 42)
for r in range(8, s_age.max_row + 1):
    code = str(s_age.cell(r, 1).value or "").strip()
    if not code or code.lower() == "none":
        continue
    if len(code) == 1:
        code = "0" + code
    
    age_40 = sum(float(s_age.cell(r, c).value or 0) for c in cols_20_74_40)
    pop_tot = pop_data.get(code, {}).get("pop_2040", 0)
    part_40 = round((age_40 / pop_tot) * 100, 1) if pop_tot > 0 else 0
    
    if code in pop_data:
        pop_data[code]["age_20_74_2040"] = int(age_40)
        pop_data[code]["part_20_74_2040"] = part_40

# -------------------------------------------------------------
# 3. Fusion avec le GeoJSON des départements
# -------------------------------------------------------------
print(f"Chargement du GeoJSON : {GEOJSON_PATH}")
with open(GEOJSON_PATH, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

for feat in geojson_data["features"]:
    code = feat["properties"].get("code", "")
    if code in pop_data:
        feat["properties"]["nom"] = pop_data[code]["nom"]
        feat["properties"]["stats"] = pop_data[code]
    else:
        feat["properties"]["stats"] = None

geojson_str = json.dumps(geojson_data, ensure_ascii=False)

# -------------------------------------------------------------
# 4. Génération de l'application HTML interactive Leaflet
# -------------------------------------------------------------
html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
    <title>Démographie France (2026-2040 & Tranche 20-74 ans)</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
    <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
    <style>
        html, body {{ width:100%; height:100%; margin:0; padding:0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; }}
        #map {{ position:absolute; top:0; bottom:0; right:0; left:0; }}
        .leaflet-popup-content {{ font-size:13px; line-height:1.55; color:#1e293b; margin:12px 14px; }}
        .popup-title {{ font-size:14px; font-weight:700; color:#0f172a; border-bottom:1px solid #e2e8f0; padding-bottom:4px; margin-bottom:6px; }}
        .leaflet-control-layers {{ background:rgba(255,255,255,0.95)!important; box-shadow:0 2px 10px rgba(0,0,0,0.15)!important; border-radius:8px!important; padding:8px 12px!important; font-size:13px!important; font-weight:600!important; }}
        .info.legend {{ background:rgba(255,255,255,0.95); padding:10px 14px; box-shadow:0 2px 10px rgba(0,0,0,0.15); border-radius:8px; font-size:12px; color:#334155; line-height:18px; max-width:250px; }}
        .legend-title {{ font-weight:700; font-size:12px; color:#0f172a; margin-bottom:6px; text-transform:uppercase; letter-spacing:0.4px; }}
        .legend-item {{ display:flex; align-items:center; gap:8px; margin-bottom:3px; }}
        .legend-color {{ width:16px; height:14px; border-radius:3px; opacity:0.75; border:1px solid rgba(0,0,0,0.2); display:inline-block; }}
        .map-title-box {{ position:absolute; top:12px; left:55px; z-index:1000; background:rgba(255,255,255,0.95); padding:8px 14px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.15); }}
        .map-title-box h1 {{ font-size:15px; font-weight:700; color:#0f172a; margin:0; }}
        .map-title-box p {{ font-size:11px; color:#64748b; margin:2px 0 0 0; }}
    </style>
</head>
<body>
    <div class="map-title-box">
        <h1>Démographie &amp; Projections par Département</h1>
        <p>Source Insee Omphale 2010 • Scénario Central</p>
    </div>
    <div id="map"></div>
    <script>
        const geojsonData = {geojson_str};
        function fmt(n) {{ if (!n && n !== 0) return '-'; return new Intl.NumberFormat('fr-FR').format(n); }}
        const map = L.map("map", {{ center:[46.6,2.2], zoom:6, zoomControl:true }});
        L.control.scale({{ metric:true, imperial:false }}).addTo(map);
        
        // Fond de carte ESRI sans clé API requise
        L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{{z}}/{{y}}/{{x}}", {{
            maxZoom:19, attribution:'Tiles &copy; Esri'
        }}).addTo(map);

        function getColorPop(evol) {{
            if (evol===undefined||evol===null) return '#94a3b8';
            if (evol < -3) return '#b91c1c';
            if (evol < 0) return '#ea580c';
            if (evol < 3) return '#64748b';
            if (evol < 7) return '#0284c7';
            if (evol < 12) return '#16a34a';
            return '#15803d';
        }}

        function getColorAge(eff) {{
            if (!eff) return '#94a3b8';
            if (eff < 150000) return '#64748b';
            if (eff < 300000) return '#6366f1';
            if (eff < 500000) return '#0284c7';
            if (eff < 800000) return '#0d9488';
            if (eff < 1200000) return '#16a34a';
            return '#ea580c';
        }}

        // Calque 1 : Évolution Population 2026-2040
        const popLayer = L.geoJson(geojsonData, {{
            style: f => {{
                const e = f.properties.stats?.evol_pop_pct || 0;
                return {{ fillColor: getColorPop(e), color: getColorPop(e), weight: 1, opacity: 1, fillOpacity: 0.52 }};
            }},
            onEachFeature: (feature, layer) => {{
                const s = feature.properties.stats;
                const nom = (s && s.nom) ? s.nom : feature.properties.code;
                const code = feature.properties.code;
                if(s) {{
                    const sign = s.evol_pop_pct >= 0 ? '+' : '';
                    layer.bindPopup(
                        `<div class="popup-title"><b>${{nom}} (${{code}})</b></div>` +
                        `<b>Population 2026 :</b> ${{fmt(s.pop_2026)}} hab.<br>` +
                        `<b>Population 2040 :</b> ${{fmt(s.pop_2040)}} hab.<br>` +
                        `<b>Évolution :</b> <span style="font-weight:700;color:${{s.evol_pop_pct >= 0 ? '#15803d' : '#b91c1c'}}">${{sign}}${{s.evol_pop_pct}}% (${{sign}}${{fmt(s.evol_pop_abs)}} hab.)</span><br>` +
                        `<hr style="margin:6px 0;border:0;border-top:1px dashed #cbd5e1;">` +
                        `<b>20-74 ans (2040) :</b> ${{fmt(s.age_20_74_2040)}} hab. (${{s.part_20_74_2040}}%)`,
                        {{ maxWidth: 300 }}
                    );
                    layer.bindTooltip(`<b>${{nom}} (${{code}}) :</b> ${{sign}}${{s.evol_pop_pct}}%`, {{ sticky: true }});
                }}
            }}
        }}).addTo(map);

        // Calque 2 : Taille brute de la tranche 20-74 ans en 2040
        const ageLayer = L.geoJson(geojsonData, {{
            style: f => {{
                const e = f.properties.stats?.age_20_74_2040 || 0;
                return {{ fillColor: getColorAge(e), color: getColorAge(e), weight: 1, opacity: 1, fillOpacity: 0.52 }};
            }},
            onEachFeature: (feature, layer) => {{
                const s = feature.properties.stats;
                const nom = (s && s.nom) ? s.nom : feature.properties.code;
                const code = feature.properties.code;
                if(s) {{
                    layer.bindPopup(
                        `<div class="popup-title"><b>${{nom}} (${{code}})</b></div>` +
                        `<b>20-74 ans (2040) :</b> <span style="font-weight:700;color:#0284c7;">${{fmt(s.age_20_74_2040)}} hab.</span><br>` +
                        `<b>Part :</b> ${{s.part_20_74_2040}}%<br>` +
                        `<b>Pop. totale 2040 :</b> ${{fmt(s.pop_2040)}} hab.`,
                        {{ maxWidth: 300 }}
                    );
                    layer.bindTooltip(`<b>${{nom}} (${{code}}) :</b> ${{fmt(s.age_20_74_2040)}} hab. (20-74 ans)`, {{ sticky: true }});
                }}
            }}
        }});

        // Sélecteur de couches
        L.control.layers(
            {{ "Évolution Population (2026-2040)": popLayer, "Taille 20-74 ans (2040)": ageLayer }},
            null,
            {{ position: 'topright', collapsed: false }}
        ).addTo(map);

        // Légende dynamique
        const legend = L.control({{ position: 'bottomright' }});
        legend.onAdd = function() {{
            const div = L.DomUtil.create('div', 'info legend');
            div.id = 'legend-content';
            updateLegend(div, 'pop');
            return div;
        }};
        legend.addTo(map);

        function updateLegend(container, mode) {{
            if (!container) container = document.getElementById('legend-content');
            if (mode === 'pop') {{
                container.innerHTML = `
                    <div class="legend-title">Évolution Pop. (2026-2040)</div>
                    <div class="legend-item"><span class="legend-color" style="background:#15803d;"></span> > +12%</div>
                    <div class="legend-item"><span class="legend-color" style="background:#16a34a;"></span> +7% à +12%</div>
                    <div class="legend-item"><span class="legend-color" style="background:#0284c7;"></span> +3% à +7%</div>
                    <div class="legend-item"><span class="legend-color" style="background:#64748b;"></span> 0% à +3%</div>
                    <div class="legend-item"><span class="legend-color" style="background:#ea580c;"></span> -3% à 0%</div>
                    <div class="legend-item"><span class="legend-color" style="background:#b91c1c;"></span> < -3%</div>`;
            }} else {{
                container.innerHTML = `
                    <div class="legend-title">Taille 20-74 ans (2040)</div>
                    <div class="legend-item"><span class="legend-color" style="background:#ea580c;"></span> > 1 200 000 hab.</div>
                    <div class="legend-item"><span class="legend-color" style="background:#16a34a;"></span> 800k - 1 200k</div>
                    <div class="legend-item"><span class="legend-color" style="background:#0d9488;"></span> 500k - 800k</div>
                    <div class="legend-item"><span class="legend-color" style="background:#0284c7;"></span> 300k - 500k</div>
                    <div class="legend-item"><span class="legend-color" style="background:#6366f1;"></span> 150k - 300k</div>
                    <div class="legend-item"><span class="legend-color" style="background:#64748b;"></span> < 150 000 hab.</div>`;
            }}
        }}

        map.on('baselayerchange', e => {{
            updateLegend(null, e.name.includes("Évolution") ? 'pop' : 'age');
        }});
    </script>
</body>
</html>"""

# -------------------------------------------------------------
# 5. Écriture du fichier HTML dans gold/maps
# -------------------------------------------------------------
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"✅ Carte générée avec succès dans Fabric : {OUTPUT_HTML}")
